# ML Systems Design & Problem Framing

**Course:** [ML in Practice](https://ml-viz-ruby.vercel.app/courses/ml-in-practice/04-ml-systems-design)

This notebook is the *framing* counterpart to the lesson. We walk four small simulations — no real model, all NumPy — that surface the four core systems-design ideas:

1. **Goodhart simulator.** A recommender's true reward is user retention; the trainable proxy is click-through rate. We show that training on CTR drives retention *down* even as CTR climbs.
2. **Composite objective.** Re-train with $L = -\text{CTR} + \lambda \cdot \text{clickbait-penalty}$ for several $\lambda$ values and plot the resulting CTR / retention Pareto frontier.
3. **Latency budget.** Simulate 10,000 requests with log-normal latencies. Compute p50 / p95 / p99 and the SLO-violation rate, then add a 50 ms feature-fetch tax to show how the upstream cost dominates the tail.
4. **Cost-vs-accuracy Pareto.** Five candidate models with different (cost, accuracy) trade-offs; identify the Pareto-efficient set.

Self-contained: NumPy + matplotlib only. No sklearn, no API keys, no network.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#2a2d3a',
    'axes.labelcolor': '#e2e8f0',
    'text.color': '#e2e8f0',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'grid.color': '#2a2d3a',
    'grid.alpha': 0.5,
})

BRAND  = '#6366f1'
TEAL   = '#2dd4bf'
ROSE   = '#fb7185'
ORANGE = '#f97316'
YELLOW = '#facc15'
MUTED  = '#475569'

## 1. Goodhart simulator: training on CTR can hurt retention

We simulate a tiny world of $N$ items. Each item has two latent attributes:

- `quality` $\in [0, 1]$: probability the user is still on the platform 30 days later given they consumed this item.
- `clickbait` $\in [0, 1]$: probability the user *clicks* on the item from the feed.

The *true* reward we want to maximise is **retention** — driven by `quality`. The *trainable* proxy is **CTR** — driven by `clickbait`. In our synthetic world the two attributes are weakly *anti-correlated*: high-clickbait items are systematically slightly lower quality. (This matches the empirical pattern in real recommender systems.)

The 'model' is a simple linear ranker over a 2-D feature vector `(quality_signal, clickbait_signal)`. Training on CTR means doing gradient ascent on a scoring weight that pushes the ranker toward the clickbait signal. We watch what happens to retention as we train.

In [ ]:
rng = np.random.default_rng(7)
N = 5000

# Latent attributes, weakly anti-correlated.
quality   = rng.beta(5, 2, size=N)
clickbait = np.clip(rng.beta(2, 5, size=N) + 0.25 * (1 - quality), 0, 1)

# Observed signals are noisy versions of the latents.
quality_signal   = np.clip(quality   + 0.10 * rng.standard_normal(N), 0, 1)
clickbait_signal = np.clip(clickbait + 0.10 * rng.standard_normal(N), 0, 1)

print(f'corr(quality, clickbait) = {np.corrcoef(quality, clickbait)[0,1]:+.3f}')
print(f'mean quality   = {quality.mean():.3f}')
print(f'mean clickbait = {clickbait.mean():.3f}')

In [ ]:
def rank_top_k(scores, k):
    '''Return indices of the top-k items by score.'''
    return np.argpartition(-scores, k)[:k]

K = 50  # feed length: top-K items shown per step

# Ranker score: w_quality * quality_signal + w_clickbait * clickbait_signal.
# We *fix* w_quality at 1.0 and let w_clickbait grow as the model 'learns' that
# clickbait predicts CTR. This mimics gradient ascent on a CTR objective.
w_clickbait_path = np.linspace(0.0, 3.0, 31)

ctr_curve       = []
retention_curve = []

for w in w_clickbait_path:
    scores = 1.0 * quality_signal + w * clickbait_signal
    top = rank_top_k(scores, K)
    # CTR of the surfaced feed = average clickbait of top-K (these are clicked).
    ctr_curve.append(clickbait[top].mean())
    # Retention = average quality of top-K (only consumed items drive retention).
    retention_curve.append(quality[top].mean())

fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(w_clickbait_path, ctr_curve, color=ROSE, linewidth=2, label='CTR (proxy, trained)')
ax1.plot(w_clickbait_path, retention_curve, color=TEAL, linewidth=2, label='retention (true goal)')
ax1.set_xlabel('clickbait weight in the ranker (training step)')
ax1.set_ylabel('mean over top-K feed')
ax1.set_title('Goodhart: CTR rises, retention falls')
ax1.grid(True)
ax1.legend(loc='center right', frameon=False)
plt.tight_layout(); plt.show()

print(f'Starting CTR        = {ctr_curve[0]:.3f},  starting retention = {retention_curve[0]:.3f}')
print(f'Final    CTR        = {ctr_curve[-1]:.3f},  final    retention = {retention_curve[-1]:.3f}')

Training on CTR works *exactly as designed*: CTR climbs monotonically. The problem is that the trained-for proxy and the true goal are not the same axis. Retention falls as the ranker leans harder on the clickbait signal. **The model is succeeding at its objective and failing at the business.** This is Goodhart's law made concrete — the same dynamic that drives reward hacking in RLHF, clickbait in news feeds, and ad-relevance gaming. The fix is not to train harder; it is to change the objective.

## 2. Composite objective: trade off CTR against a clickbait penalty

The Goodhart fix is to make the objective *say what we mean*. We introduce a composite objective:

$$ L(w) = -\text{CTR}(w) + \lambda \cdot \overline{\text{clickbait}}(w), $$

where $w$ is the clickbait weight in the ranker and $\overline{\text{clickbait}}$ is the mean clickbait score of the surfaced feed. Larger $\lambda$ means we penalise the clickbait component more. We sweep $\lambda \in \{0, 0.2, 0.5, 1.0\}$, find the optimal $w$ for each, and trace the resulting (CTR, retention) frontier.

In [ ]:
lambdas = [0.0, 0.2, 0.5, 1.0]

# For each lambda, search over w_clickbait and pick the w that minimises the
# composite loss. Then record (CTR, retention) at that optimal w.
points = []
for lam in lambdas:
    losses = []
    for w in w_clickbait_path:
        scores = 1.0 * quality_signal + w * clickbait_signal
        top = rank_top_k(scores, K)
        ctr = clickbait[top].mean()
        penalty = clickbait_signal[top].mean()
        losses.append(-ctr + lam * penalty)
    losses = np.array(losses)
    w_star = w_clickbait_path[np.argmin(losses)]
    scores = 1.0 * quality_signal + w_star * clickbait_signal
    top = rank_top_k(scores, K)
    points.append({
        'lambda':    lam,
        'w_star':    float(w_star),
        'ctr':       float(clickbait[top].mean()),
        'retention': float(quality[top].mean()),
    })

print(f'{"lambda":>6s}  {"w*":>6s}  {"CTR":>6s}  {"retention":>10s}')
for p in points:
    print(f'{p["lambda"]:>6.2f}  {p["w_star"]:>6.2f}  {p["ctr"]:>6.3f}  {p["retention"]:>10.3f}')

# Plot the Pareto frontier.
ctr_pts = [p['ctr'] for p in points]
ret_pts = [p['retention'] for p in points]
fig, ax = plt.subplots(figsize=(7.5, 4.3))
ax.plot(ctr_pts, ret_pts, '-o', color=BRAND, linewidth=2, markersize=9)
for p in points:
    ax.annotate(f' lambda={p["lambda"]:.1f}',
                xy=(p['ctr'], p['retention']),
                color='#e2e8f0', fontsize=9)
ax.set_xlabel('CTR (proxy)')
ax.set_ylabel('retention (true goal)')
ax.set_title('Composite-objective Pareto frontier')
ax.grid(True)
plt.tight_layout(); plt.show()

The frontier slopes down-and-right: every step that wins extra CTR costs retention. Picking $\lambda$ is a *business call*, not a statistical one — how much CTR is the team willing to give up for a healthier long-term funnel? That decision lives with the product leadership, but the engineering job is to (1) construct the frontier so the trade-off is visible, and (2) report *both* axes — never just the trained-for one.

## 3. Latency budget: the tail is what users feel

A model's median latency tells you very little about user experience. The *tail* — p95, p99 — is what drives perceived reliability. Below we simulate 10,000 inference requests from a log-normal latency distribution (a realistic shape for ML model serving on a shared GPU). We then compute the SLO violation rate against a 200 ms budget, and finally we add a 50 ms 'feature fetch' tax to show how an upstream feature-store call can dominate the latency budget even though it never appears in the model's own profiling.

In [ ]:
rng = np.random.default_rng(13)
N = 10_000

# Log-normal latencies in milliseconds. mu and sigma in log-space.
# Tuned so median ~80 ms and tail extends out past 300 ms.
latencies_model = rng.lognormal(mean=np.log(80), sigma=0.45, size=N)

p50  = np.percentile(latencies_model, 50)
p95  = np.percentile(latencies_model, 95)
p99  = np.percentile(latencies_model, 99)
SLO  = 200.0
violate = (latencies_model > SLO).mean()

print(f'Model-only latencies on {N} requests:')
print(f'  p50         = {p50:.1f} ms')
print(f'  p95         = {p95:.1f} ms')
print(f'  p99         = {p99:.1f} ms')
print(f'  > {SLO:.0f} ms SLO = {100*violate:.1f}% of requests')

# Now add a 50 ms feature-fetch tax.
latencies_full = latencies_model + 50.0
violate_full = (latencies_full > SLO).mean()
p95_full = np.percentile(latencies_full, 95)
p99_full = np.percentile(latencies_full, 99)

print(f'\nAfter +50 ms feature-fetch tax:')
print(f'  p95         = {p95_full:.1f} ms')
print(f'  p99         = {p99_full:.1f} ms')
print(f'  > {SLO:.0f} ms SLO = {100*violate_full:.1f}% of requests')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(latencies_model, bins=80, range=(0, 400), color=BRAND, alpha=0.7, label='model only')
ax.hist(latencies_full,  bins=80, range=(0, 400), color=ROSE,  alpha=0.5, label='model + feature fetch')
ax.axvline(SLO, color=YELLOW, linestyle='--', label=f'SLO = {SLO:.0f} ms')
ax.axvline(p95, color=BRAND, linestyle=':', alpha=0.8, label=f'model p95 = {p95:.0f} ms')
ax.axvline(p95_full, color=ROSE, linestyle=':', alpha=0.8, label=f'full p95 = {p95_full:.0f} ms')
ax.set_xlabel('latency (ms)')
ax.set_ylabel('request count')
ax.set_title('Latency distribution: model vs model + feature-fetch tax')
ax.legend(loc='upper right', frameon=False, fontsize=9)
ax.grid(True, axis='y')
plt.tight_layout(); plt.show()

Two lessons from this simulation:

1. **The mean is misleading.** The mean of a log-normal latency distribution is well *below* the p95. A team that reports 'average latency = 95 ms' for an SLO of 200 ms feels safe — until they realise 1 in 20 users sees over 200 ms.
2. **Upstream costs dominate the tail.** A 50 ms feature-fetch tax — invisible to the *model*'s profiler — pushes a large fraction of borderline requests past the SLO. Optimising the model's own forward pass in this regime would buy single-digit milliseconds; optimising the feature fetch buys 50 ms straight off the tail. **Profile the *whole* request path, not just the model.**

## 4. Cost-vs-accuracy Pareto frontier

Given five candidate models with different (cost, accuracy) pairs, identify the **Pareto-efficient** set — the models that are not dominated by any other model on both axes. A model is *dominated* if there exists another model that is at least as accurate *and* at least as cheap, with at least one strict inequality.

The Pareto-efficient set is the shortlist a model-selection process should consider. Everything else is strictly worse on the cost / accuracy axes and only worth shipping if it wins on a third dimension (privacy, latency, refusal rate) that we haven't modelled here.

In [ ]:
# (cost per query in $, accuracy on the eval set)
candidates = [
    ('A',  0.0001, 0.78),
    ('B',  0.0040, 0.85),
    ('C',  0.0070, 0.86),  # dominated by B (higher cost, only +1pt accuracy)
    ('D',  0.0420, 0.92),
    ('E',  0.0450, 0.91),  # dominated by D (higher cost, lower accuracy)
]

def pareto_front(points):
    '''Return the indices of non-dominated points where lower cost and higher
    accuracy are both preferred. points is a list of (cost, accuracy) tuples.'''
    keep = []
    for i, (ci, ai) in enumerate(points):
        dominated = False
        for j, (cj, aj) in enumerate(points):
            if i == j:
                continue
            if cj <= ci and aj >= ai and (cj < ci or aj > ai):
                dominated = True
                break
        if not dominated:
            keep.append(i)
    return keep

pts = [(c, a) for (_, c, a) in candidates]
front_idx = pareto_front(pts)
front_names = [candidates[i][0] for i in front_idx]
print(f'Pareto-efficient models: {front_names}')

fig, ax = plt.subplots(figsize=(8, 4.3))
for i, (name, cost, acc) in enumerate(candidates):
    color = TEAL if i in front_idx else ROSE
    label = 'Pareto-efficient' if i in front_idx else 'dominated'
    ax.scatter([cost], [acc], color=color, s=140, edgecolor='#0f1117', zorder=3)
    ax.annotate(f' {name}', xy=(cost, acc), color='#e2e8f0', fontsize=11)

# Connect the Pareto front from cheapest to most expensive.
front_sorted = sorted([pts[i] for i in front_idx], key=lambda p: p[0])
ax.plot([p[0] for p in front_sorted], [p[1] for p in front_sorted],
        '--', color=TEAL, alpha=0.6, linewidth=2, label='Pareto frontier')

ax.set_xscale('log')
ax.set_xlabel('cost per query ($, log scale)')
ax.set_ylabel('accuracy')
ax.set_title('Cost-vs-accuracy: keep only the Pareto-efficient candidates')
ax.legend(loc='lower right', frameon=False)
ax.grid(True, which='both')
plt.tight_layout(); plt.show()

The Pareto-efficient set excludes models C and E: C is strictly worse than B (more expensive, barely more accurate), and E is strictly worse than D (more expensive, less accurate). The remaining three (A, B, D) span the cost / accuracy axis and each one wins for some traffic mix. The next decision — *which* of A, B, D to ship — is a question of business priorities (cost ceiling, minimum acceptable accuracy) and is what the routing-tier pattern from the [Evaluating AI Systems](https://ml-viz-ruby.vercel.app/courses/model-evaluation/05-evaluating-ai-systems) lesson is built to address.

---
## ✏️ Your turn

### Exercise: implement `pareto_front(points)`

Given a list `points` of `(cost, accuracy)` tuples, return the **list of indices** of the non-dominated subset (the Pareto frontier), where lower cost and higher accuracy are both preferred.

A point $p_i = (c_i, a_i)$ is *dominated* by $p_j = (c_j, a_j)$ when $c_j \le c_i$ and $a_j \ge a_i$, with at least one strict inequality.

The tests below check:

1. The 5-point reference set from §4 returns the expected `[0, 1, 3]` (A, B, D).
2. A single-point set returns `[0]`.
3. A set where every point lies on the frontier returns all indices.
4. A set where one point dominates everything returns just that point's index.

In [ ]:
def pareto_front(points):
    '''Return the indices of non-dominated points.

    Args:
        points: list of (cost, accuracy) tuples. Lower cost is better;
                higher accuracy is better.

    Returns:
        A list of int indices (in any order) of the points that are
        not strictly dominated by any other point.
    '''
    # TODO(you):
    #   For each point i, check every other point j. If any j has
    #     cost_j <= cost_i AND accuracy_j >= accuracy_i
    #     with at least one strict inequality, then i is dominated.
    #   Keep only the indices that are NOT dominated.
    pass

# Smoke run on the 5-point reference set.
ref = [(0.0001, 0.78), (0.0040, 0.85), (0.0070, 0.86), (0.0420, 0.92), (0.0450, 0.91)]
print(f'pareto_front(ref) = {pareto_front(ref)}')

In [ ]:
# Test 1: 5-point reference set from §4 -- expected [0, 1, 3].
ref = [(0.0001, 0.78), (0.0040, 0.85), (0.0070, 0.86), (0.0420, 0.92), (0.0450, 0.91)]
out = pareto_front(ref)
assert out is not None, 'pareto_front returned None'
assert sorted(out) == [0, 1, 3], f'expected [0, 1, 3], got {sorted(out)}'

# Test 2: single point is always on the frontier.
assert sorted(pareto_front([(0.01, 0.8)])) == [0]

# Test 3: every point lies on the frontier (each is unique on cost+accuracy).
front = [(0.001, 0.70), (0.005, 0.80), (0.010, 0.85), (0.050, 0.90)]
assert sorted(pareto_front(front)) == [0, 1, 2, 3]

# Test 4: one point dominates everything else.
dom = [(0.001, 0.99), (0.005, 0.80), (0.010, 0.70), (0.050, 0.60)]
assert sorted(pareto_front(dom)) == [0]

# Test 5: ties on accuracy but cheaper cost wins.
ties = [(0.001, 0.85), (0.005, 0.85), (0.010, 0.85)]
assert sorted(pareto_front(ties)) == [0]

print('All Pareto-front tests passed.')

<details>
<summary>Show solution</summary>

```python
def pareto_front(points):
    keep = []
    for i, (ci, ai) in enumerate(points):
        dominated = False
        for j, (cj, aj) in enumerate(points):
            if i == j:
                continue
            # j dominates i iff j is at least as good on both axes
            # and strictly better on at least one.
            if cj <= ci and aj >= ai and (cj < ci or aj > ai):
                dominated = True
                break
        if not dominated:
            keep.append(i)
    return keep
```

Three things worth noticing:

1. **The double loop is $O(n^2)$.** Fine for a model shortlist (typically $n \le 10$). For thousands of candidates, sort by cost and sweep maintaining a running max of accuracy in $O(n \log n)$.
2. **The strict-inequality clause matters.** Without it, two identical points would dominate each other and both would be dropped. The 'at least one strict inequality' rule keeps the frontier well-defined under duplicates.
3. **The frontier is the shortlist, not the answer.** Picking *among* the Pareto-efficient models is a separate decision that depends on the cost ceiling, the minimum acceptable accuracy, and any third-axis constraints (privacy, latency, refusal rate) the cost/accuracy plot does not capture.
</details>